# Validate against buoy observations

Compare **offshore hindcast** (closest goal node), **bulk metamodel**, **spectral metamodel**, and **dynamic SnapWave** to buoy records.

**Prerequisites**

1. Fitted models — `06_Validate_Metamodel.ipynb` (`mm.fit_all`)
2. Dynamic run for the same period — `08_Dynamic_Cases_Generation.ipynb`
3. Observations GeoJSON — `03_Generate_Observations_Configuration.ipynb` → `inputs/Observations/<Case>_observations.json`
4. Processed buoy file(s) under `inputs/Observations/processed/` (you build these in the notebook below)

Use the **same `period`** for buoy data, reconstruction, and dynamic output.

Comparisons request **`hs`, `tp`, `dir`, and `spr`**. Variables missing from the buoy
file or any model are skipped automatically.

In [1]:
import time

import matplotlib.pyplot as plt
import xarray as xr
from utils.config import SIMULATIONS, resolve
from utils import validation as val
from utils import metamodel as mm
from utils import onshore_sites as obs

## Configuration

Auto-resolves every buoy with real spectral observations under
`inputs/Observations/processed/spectra/` (currently 3 of 35 — see notebook 11) and
validates all of them together, pooled across every year they have data for. Point
`spectral_buoy_dir` elsewhere, or hand-pick `buoy_instruments`, to validate a
different set.

In [2]:
period = slice("2001", "2011")  # years all 3 spectral buoys have real observations
period_label = "2001-2011"
offshore_name = "WHACS"
mda_n = 450  # must match a fitted mda_candidates entry
pca_variance = 0.99

sites = ["buoy_BatemansBay_Buoy", "buoy_ByronBay_Buoy", "buoy_Sydney_Buoy"]

# Processed observation NetCDF (spectra) per SnapWave site id
buoy_raw_files = {
    "buoy_BatemansBay_Buoy": resolve(
        "inputs/Observations/processed/spectra/BatemansBay_Buoy.nc"
    ),
    "buoy_ByronBay_Buoy": resolve(
        "inputs/Observations/processed/spectra/ByronBay_Buoy.nc"
    ),
    "buoy_Sydney_Buoy": resolve("inputs/Observations/processed/Sydney_Buoy.nc"),
}

requested_variables = ["hs", "tp", "dir", "spr"]

dynamic_path = (
    resolve(SIMULATIONS["dynamic_cases"])
    / f"Dynamic_Cases_{offshore_name}_{period_label}.nc"
)

print("Observations JSON:", obs.geojson_path())
print(f"period         : {period_label}")
print(f"mda_n          : {mda_n}")
print(f"sites          : {sites}")
print(f"requested vars : {requested_variables}")
print(f"dynamic path   : {dynamic_path}")
print(f"dynamic exists : {dynamic_path.is_file()}")

Observations JSON: /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/inputs/Observations/NewSouthWales_observations.json
period         : 2001-2011
mda_n          : 450
sites          : ['buoy_BatemansBay_Buoy', 'buoy_ByronBay_Buoy', 'buoy_Sydney_Buoy']
requested vars : ['hs', 'tp', 'dir', 'spr']
dynamic path   : /hyglowdata/NewSouthWales/SnapWave_CASES_Dynamic/Dynamic_Cases_WHACS_2001-2011.nc
dynamic exists : False


## 1 · Check models

In [3]:
mm.require_models_all(mda_n, pca_variance=pca_variance)

## 2 · Process buoy observations

Load each processed file, rename variables to HyGloW names (`hs`, `tp`, `dir`, `spr`),
slice to `period`, and build a validation dataset on `(time, sites)` with CF-style
attributes. Site metadata (lon/lat/depth) comes from the observations GeoJSON.

In [4]:
# CF-style attributes for validation plots
_BUOY_VAR_ATTRS = {
    "hs": {"standard_name": "sea_surface_wave_significant_height", "units": "m"},
    "tp": {"standard_name": "sea_surface_wave_period", "units": "s"},
    "dir": {"standard_name": "sea_surface_wave_from_direction", "units": "degree"},
    "spr": {
        "standard_name": "sea_surface_wave_directional_spread",
        "units": "degree",
    },
}

chunks: list[xr.Dataset] = []
for site_id in sites:
    raw = xr.open_dataset(buoy_raw_files[site_id]).sel(time=period)

    if "Efth" in raw or "efth" in raw:
        spec_name = "Efth" if "Efth" in raw else "efth"
        stats = raw[spec_name].spec.stats(["hs", "tp", "dm", "dspr"]).load()
        site_ds = stats.rename({"dm": "dir", "dspr": "spr"})
    else:
        rename = {}
        if "dspr" in raw.data_vars and "spr" not in raw.data_vars:
            rename["dspr"] = "spr"
        if rename:
            raw = raw.rename(rename)
        vars_out = {}
        for name in requested_variables:
            if name in raw.data_vars:
                vars_out[name] = raw[name]
            elif name == "dir" and "dm" in raw.data_vars:
                vars_out["dir"] = raw["dm"]
        if "hs" not in vars_out:
            raise ValueError(f"{site_id}: processed buoy file must contain hs or Efth")
        site_ds = xr.Dataset(vars_out)

    site_ds = site_ds.expand_dims(sites=[site_id]).transpose("time", "sites")
    for var, attrs in _BUOY_VAR_ATTRS.items():
        if var in site_ds:
            site_ds[var].attrs.update(attrs)
    chunks.append(obs.attach_buoy_site_meta(site_ds, site_id))

buoy = xr.concat(chunks, dim="sites") if len(chunks) > 1 else chunks[0]
buoy

<xarray.Dataset> Size: 16MB
Dimensions:         (time: 152323, sites: 3)
Coordinates:
  * time            (time) datetime64[ns] 1MB 2001-01-01 ... 2011-12-31T23:00...
  * sites           (sites) object 24B 'buoy_BatemansBay_Buoy' ... 'buoy_Sydn...
    site_source     (sites) <U4 48B 'buoy' 'buoy' 'buoy'
    site_label      (sites) <U16 192B 'BatemansBay_Buoy' ... 'Sydney_Buoy'
    site_longitude  (sites) float64 24B 150.3 153.7 151.4
    site_latitude   (sites) float64 24B -35.73 -28.75 -33.77
    site_depth      (sites) float64 24B -75.0 -65.0 -85.0
Data variables:
    hs              (time, sites) float64 4MB nan 2.401 1.36 ... nan nan 1.219
    tp              (time, sites) float64 4MB nan 8.199 11.1 ... nan nan 9.32
    dir             (time, sites) float64 4MB nan 154.0 174.0 ... nan nan 117.0
    spr             (time, sites) float64 4MB nan 38.52 nan nan ... nan nan nan
Attributes:
    standard_name:  sea_surface_wave_significant_height
    units:          m

## 3 · Reconstruct metamodel, load dynamic, and closest-goal offshore hindcast

Metamodel bulk/spectral reconstructions, dynamic SnapWave, and bulk offshore forcing
at each buoy's **closest goal hindcast node** (all sources in the case JSON —
currently WHACS only).

In [5]:
timings: dict[str, float] = {}

t0 = time.perf_counter()
bulk = mm.reconstruct(
    period,
    mda_n,
    spectral=False,
    offshore_name=offshore_name,
    sites=sites,
    pca_variance=pca_variance,
)
timings["Bulk metamodel"] = time.perf_counter() - t0

t0 = time.perf_counter()
spectral = mm.reconstruct(
    period,
    mda_n,
    spectral=True,
    offshore_name=offshore_name,
    sites=sites,
    pca_variance=pca_variance,
)
timings["Spectral metamodel"] = time.perf_counter() - t0
# Near-zero above means these sites/period were already cached from a
# previous run, not a fresh-reconstruction cost. Offshore hindcast below
# is a plain lookup, not a reconstruction, so it has no timing entry.

# if not dynamic_path.is_file():
#     raise FileNotFoundError(
#         f"Dynamic output not found: {dynamic_path}\n"
#         "Run 08_Dynamic_Cases_Generation for this period or change `period`."
#     )

# dynamic = xr.open_dataset(dynamic_path).sel(time=period)
# site_ids = mm.resolve_reconstruction_sites(dynamic, sites)
# dynamic = dynamic.sel(sites=site_ids).load()

offshore_hindcast, closest_goals = val.load_offshore_hindcast_at_buoys(
    period,
    sites,
    variables=requested_variables,
)
closest_goals

estimates = {
    **offshore_hindcast,
    "Bulk metamodel": bulk,
    "Spectral metamodel": spectral,
    # "Dynamic SnapWave": dynamic,
}

variables = val.resolve_comparison_variables(
    buoy,
    *estimates.values(),
    variables=requested_variables,
)
print(f"Comparing: {variables}")

Comparing: ['hs', 'tp', 'dir', 'spr']


## 4 · Metrics vs buoy

In [6]:
results = val.compare_all(
    buoy,
    estimates,
    sites=sites,
    variables=variables,
    reference_label="Buoy",
    pca_variance=pca_variance,
)

summary = val.metrics_summary(results)
summary

,pca_variances,group,site_source,site_label,variable,label,n,bias,rmse,r2,estimate_label,reference_label,model
0,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_BatemansBay_Buoy,buoy,waves,hs,Hs,83620,0.294765,0.586998,0.440080,WHACS (closest goal),Buoy,WHACS (closest goal)
1,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_BatemansBay_Buoy,buoy,waves,tp,Tp,83620,-0.635089,2.381966,-0.015045,WHACS (closest goal),Buoy,WHACS (closest goal)
2,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_BatemansBay_Buoy,buoy,waves,dir,Dir,83612,-0.109700,43.567400,-1.000988,WHACS (closest goal),Buoy,WHACS (closest goal)
3,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_BatemansBay_Buoy,buoy,waves,spr,Spr,83620,1.694815,10.472956,0.410580,WHACS (closest goal),Buoy,WHACS (closest goal)
4,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_ByronBay_Buoy,buoy,waves,hs,Hs,65375,0.086311,0.455594,0.641700,WHACS (closest goal),Buoy,WHACS (closest goal)
5,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_ByronBay_Buoy,buoy,waves,tp,Tp,69393,-0.356621,2.161109,0.102091,WHACS (closest goal),Buoy,WHACS (closest goal)
6,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_ByronBay_Buoy,buoy,waves,dir,Dir,65280,-3.414342,33.144608,0.158232,WHACS (closest goal),Buoy,WHACS (closest goal)
7,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_ByronBay_Buoy,buoy,waves,spr,Spr,69393,0.958231,9.164847,0.468704,WHACS (closest goal),Buoy,WHACS (closest goal)
8,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_Sydney_Buoy,buoy,waves,hs,Hs,29429,0.229040,0.445867,0.668917,WHACS (closest goal),Buoy,WHACS (closest goal)
9,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_Sydney_Buoy,buoy,waves,tp,Tp,29429,-0.544121,2.272404,0.049209,WHACS (closest goal),Buoy,WHACS (closest goal)


## 4b · Save summary

In [7]:
summary_out = summary.copy()
summary_out.insert(0, "period", period_label)
summary_out.insert(1, "mda_candidate", mda_n)
summary_out.insert(2, "offshore_name", offshore_name)
summary_out["reconstruct_seconds"] = summary_out["model"].map(timings)

out_path = val.buoy_validation_path("offshore", period_label)
out_path.parent.mkdir(parents=True, exist_ok=True)
summary_out.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

Wrote /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/offshore_summary_2001-2011.csv


## 5 · Scatter plots

In [8]:
# for site in sites:
#     for name, result in results.items():
#         result.plot_scatter(
#             site,
#             variables=variables,
#             estimate_label=name,
#             reference_label="Buoy",
#             cmap="Reds",
#         )
#         plt.suptitle(f"{name} vs buoy — {site} — {period_label}")
#         plt.show()

## 6 · Time series

Line plots of buoy observations vs each model over the validation period (direction as scatter).

In [9]:
# for site in sites:
#     val.plot_buoy_timeseries(buoy, estimates, site, variables=variables)
#     plt.suptitle(f"Buoy validation — {site} — {period_label}")
#     plt.show()

## 7 · Save per-buoy summary PNG (GUI)

Density scatter (top row, buoy vs. spectral metamodel) + one wide Hs/Tp time series (bottom row, cropped to when the buoy actually has data) — one combined PNG per buoy, shown as-is in the GUI's Buoy Validation tab when you click that buoy's star on the map.

In [11]:
for site in sites:
    fig = val.plot_buoy_summary(
        buoy,
        estimates,
        site,
        variables=variables,
        reference_label="Buoy",
        period_label=period_label,
        cmap="Reds",
    )
    out_path = val.buoy_summary_figure_path(site)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"saved {out_path}")

saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_BatemansBay_Buoy.png
saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_ByronBay_Buoy.png
saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_Sydney_Buoy.png
